# Phase 11: Generative Models

## What you'll learn
- Autoencoders (AE)
- Variational Autoencoders (VAE)
- GANs — Generator + Discriminator
- DCGAN implementation
- Diffusion models basics

---

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 11.1 Autoencoder

Learns to **compress** data into a latent space (encoder) and **reconstruct** it (decoder).

```
Input (784) → Encoder → Latent (32) → Decoder → Reconstruction (784)
```

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(784, 256), nn.ReLU(),
            nn.Linear(256, latent_dim), nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256), nn.ReLU(),
            nn.Linear(256, 784), nn.Sigmoid()  # output in [0,1]
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

# Data
transform = transforms.Compose([transforms.ToTensor()])
train_set = torchvision.datasets.MNIST('./data', train=True, download=True, transform=transform)
loader = DataLoader(train_set, batch_size=128, shuffle=True)

# Train
ae = Autoencoder().to(device)
optimizer = optim.Adam(ae.parameters(), lr=1e-3)

for epoch in range(5):
    total_loss = 0
    for imgs, _ in loader:
        imgs = imgs.view(-1, 784).to(device)
        recon = ae(imgs)
        loss = F.mse_loss(recon, imgs)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: loss={total_loss/len(loader):.4f}")

In [ ]:
# Visualize reconstructions
ae.eval()
with torch.no_grad():
    sample_imgs, _ = next(iter(loader))
    sample_imgs = sample_imgs[:8].view(-1, 784).to(device)
    recon = ae(sample_imgs)

fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i in range(8):
    axes[0, i].imshow(sample_imgs[i].cpu().view(28, 28), cmap='gray')
    axes[1, i].imshow(recon[i].cpu().view(28, 28), cmap='gray')
    axes[0, i].axis('off')
    axes[1, i].axis('off')
axes[0, 0].set_ylabel('Original')
axes[1, 0].set_ylabel('Reconstructed')
plt.tight_layout()
plt.show()

## 11.2 Variational Autoencoder (VAE)

Unlike AE, VAE learns a **probability distribution** in latent space, enabling generation of new samples.

Key idea: encode to (mean, log_var), sample using **reparameterization trick**.

In [ ]:
class VAE(nn.Module):
    def __init__(self, latent_dim=16):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(784, 256), nn.ReLU())
        self.fc_mu = nn.Linear(256, latent_dim)
        self.fc_logvar = nn.Linear(256, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256), nn.ReLU(),
            nn.Linear(256, 784), nn.Sigmoid()
        )

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        h = self.encoder(x)
        mu, logvar = self.fc_mu(h), self.fc_logvar(h)
        z = self.reparameterize(mu, logvar)
        return self.decoder(z), mu, logvar

def vae_loss(recon, x, mu, logvar):
    recon_loss = F.binary_cross_entropy(recon, x, reduction='sum')
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + kl_loss

# Train
vae = VAE().to(device)
optimizer = optim.Adam(vae.parameters(), lr=1e-3)

for epoch in range(5):
    total_loss = 0
    for imgs, _ in loader:
        imgs = imgs.view(-1, 784).to(device)
        recon, mu, logvar = vae(imgs)
        loss = vae_loss(recon, imgs, mu, logvar)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: loss={total_loss/len(loader.dataset):.2f}")

In [ ]:
# Generate new images by sampling from latent space
vae.eval()
with torch.no_grad():
    z = torch.randn(8, 16).to(device)
    generated = vae.decoder(z)

fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for i in range(8):
    axes[i].imshow(generated[i].cpu().view(28, 28), cmap='gray')
    axes[i].axis('off')
plt.suptitle('VAE Generated Samples')
plt.show()

## 11.3 GAN — Generative Adversarial Network

Two networks competing:
- **Generator**: creates fake data from noise
- **Discriminator**: distinguishes real from fake

They train adversarially until the generator fools the discriminator.

In [ ]:
latent_dim = 64

class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 256), nn.LeakyReLU(0.2),
            nn.Linear(256, 512), nn.LeakyReLU(0.2),
            nn.Linear(512, 784), nn.Tanh()  # output in [-1, 1]
        )
    def forward(self, z):
        return self.net(z)

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(784, 512), nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Linear(512, 256), nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Linear(256, 1)  # raw logit
        )
    def forward(self, x):
        return self.net(x)

In [ ]:
# Normalize to [-1, 1] for Tanh generator
transform_gan = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])
gan_dataset = torchvision.datasets.MNIST('./data', train=True, download=True, transform=transform_gan)
gan_loader = DataLoader(gan_dataset, batch_size=128, shuffle=True)

G = Generator().to(device)
D = Discriminator().to(device)
opt_G = optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_D = optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))
criterion = nn.BCEWithLogitsLoss()

for epoch in range(10):
    for real_imgs, _ in gan_loader:
        batch_size = real_imgs.size(0)
        real_imgs = real_imgs.view(-1, 784).to(device)
        real_labels = torch.ones(batch_size, 1).to(device)
        fake_labels = torch.zeros(batch_size, 1).to(device)

        # --- Train Discriminator ---
        z = torch.randn(batch_size, latent_dim).to(device)
        fake_imgs = G(z).detach()

        d_real = D(real_imgs)
        d_fake = D(fake_imgs)
        d_loss = criterion(d_real, real_labels) + criterion(d_fake, fake_labels)

        opt_D.zero_grad()
        d_loss.backward()
        opt_D.step()

        # --- Train Generator ---
        z = torch.randn(batch_size, latent_dim).to(device)
        fake_imgs = G(z)
        g_loss = criterion(D(fake_imgs), real_labels)  # fool discriminator

        opt_G.zero_grad()
        g_loss.backward()
        opt_G.step()

    print(f"Epoch {epoch+1}: D_loss={d_loss.item():.4f} G_loss={g_loss.item():.4f}")

In [ ]:
# Generate samples
G.eval()
with torch.no_grad():
    z = torch.randn(8, latent_dim).to(device)
    generated = G(z)

fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for i in range(8):
    img = generated[i].cpu().view(28, 28) * 0.5 + 0.5  # denormalize
    axes[i].imshow(img, cmap='gray')
    axes[i].axis('off')
plt.suptitle('GAN Generated Samples')
plt.show()

## 11.4 Diffusion Models — Core Idea

Diffusion models work by:
1. **Forward process**: gradually add noise to data over T steps
2. **Reverse process**: train a neural network to **denoise** step by step

At inference, start from pure noise and iteratively denoise to generate images.

In [ ]:
# Simplified diffusion concept
T = 1000  # total timesteps
betas = torch.linspace(1e-4, 0.02, T)  # noise schedule
alphas = 1 - betas
alpha_cumprod = torch.cumprod(alphas, dim=0)

def add_noise(x0, t, noise=None):
    """Forward process: add noise to x0 at timestep t."""
    if noise is None:
        noise = torch.randn_like(x0)
    sqrt_alpha = alpha_cumprod[t].sqrt().view(-1, 1)
    sqrt_one_minus = (1 - alpha_cumprod[t]).sqrt().view(-1, 1)
    return sqrt_alpha * x0 + sqrt_one_minus * noise, noise

# Demo: progressively noisier images
x0 = torch.randn(1, 784)  # clean image
timesteps = [0, 100, 300, 500, 700, 999]

fig, axes = plt.subplots(1, len(timesteps), figsize=(12, 2))
for i, t in enumerate(timesteps):
    noisy, _ = add_noise(x0, torch.tensor([t]))
    axes[i].imshow(noisy.view(28, 28), cmap='gray')
    axes[i].set_title(f't={t}')
    axes[i].axis('off')
plt.suptitle('Forward Diffusion Process')
plt.show()

print("For full diffusion training, see libraries like: diffusers (Hugging Face)")

## ✅ Phase 11 Summary

You now understand:
- Autoencoders for compression and reconstruction
- VAEs for learning latent distributions and generation
- GANs with adversarial training (Generator vs Discriminator)
- Diffusion models concept (add noise → learn to denoise)

**Next → Phase 12: Debugging & Best Practices**